# Lab 3 — Results & Error Mitigation

This chapter is in two parts:

- **Part A — Noise, backends, and benchmarking.** How real hardware shapes measured
  results: reading backend information, building backend-informed noisy simulators, toy
  noise models, and comparing the Heron and Nighthawk architectures.
- **Part B — Error mitigation in Qiskit Runtime.** The families of techniques for handling
  noise, the error suppression and mitigation options on `EstimatorV2`, and the Pauli
  dressing and commutation algebra they rest on.

No QPU time is needed. Part A, Chapter 1 reads a real backend's properties with your IBM
Quantum credentials (`api.json`) and falls back to `FakeNighthawk`; everything
else runs locally on simulators and fake backends.

## Table of Contents

- [Setup](#setup): [installation](#installation), [imports](#imports), [helper functions](#helper-functions)
- **[Part A — Noise, Backends, and Benchmarking](#part-a)**
  - [Chapter 1 — Noise in quantum computers and backend-aware simulation](#part-a-ch1)
  - [Chapter 2 — What kind of noise do we care about?](#part-a-ch2)
  - [Chapter 3 — Comparing Heron and Nighthawk Architectures](#part-a-ch3)
- **[Part B — Error Mitigation in Qiskit Runtime](#part-b)**
  - [Chapter 1 — Errors on today's quantum hardware](#part-b-ch1)
  - [Chapter 2 — Error mitigation as a Runtime product, and the theory behind its cost](#part-b-ch2)
- [Additional information](#additional-information)


<a id="setup"></a><a id="installation"></a>

## Installation

If you are running this notebook in a fresh environment, uncomment and run the next cell.
It installs everything both parts need.


Install `Graviz` for smooth implementation if you are working locally; Follow the instructions [here](https://graphviz.org/download/).

In [ ]:
!pip3 install -q ipykernel "qiskit==2.4.0rc2" qiskit-aer "qiskit-ibm-runtime==0.47.0" \
    qiskit_experiments numpy scipy pandas networkx matplotlib seaborn sympy pylatexenc

<a id="imports"></a>

## Imports


In [ ]:
import numpy as np                      # For linear algebra and numerical operations
import pandas as pd                     # For data manipulation and analysis
from scipy.optimize import curve_fit    # For curve fitting and optimization
import matplotlib.pyplot as plt         # For plotting and visualization
import networkx as nx                   # For manipulating complex graphs/networks
from collections import Counter         # For counting hashable objects

from IPython.display import display     # For displaying rich content in Jupyter notebooks

from qiskit import (
    QuantumCircuit,                     # For creating and manipulating quantum circuits
    transpile,                          #
    generate_preset_pass_manager        #
)
from qiskit.quantum_info import Pauli           # For Pauli algebra in Part B
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit import pauli_twirl_2q_gates
from qiskit.visualization import plot_histogram
from qiskit.providers.exceptions import (
    QiskitBackendNotFoundError          # For handling backend not found errors
)
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel,
    ReadoutError,
    depolarizing_error,
    pauli_error,
    thermal_relaxation_error,
)
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.options import EstimatorOptions
from qiskit_ibm_runtime.fake_provider import FakeNighthawk, FakeTorino, FakeKingston
from qiskit_ibm_runtime.fake_provider import FakeProviderForBackendV2
from qiskit.visualization import plot_gate_map

<a id="helper-functions"></a>

## Helper functions

We use a few small helper functions to keep the main experiments readable.

In [ ]:
def run_counts(backend, circuit, shots=4096):
    """
    Run a quantum circuit on a given backend and return the measurement counts.

    Args:
        backend: The quantum backend to run the circuit on.
        circuit: A transpiled quantum circuit to be executed.
        shots: The number of times the circuit is executed (default is 4096).

    Returns:
        A dictionary containing the measurement counts for each possible outcome.
    """
    sampler = Sampler(mode=backend)
    job = sampler.run([circuit], shots=shots)
    result = job.result()
    cr = list(result[0].data.keys())
    if len(cr) != 1:
        raise ValueError(f"Expected exactly one circuit result, got {len(cr)}")
    counts = result[0].data[cr[0]].get_counts()
    return counts

def transpile_and_run_counts(backend, circuit, shots=4096):
    """
    Transpile a quantum circuit for a given backend and run it to obtain measurement counts.

    Args:
        backend: The quantum backend to run the circuit on.
        circuit: Raw Untranspiled quantum circuit to be executed.
        shots: The number of times the circuit is executed (default is 4096).

    Returns:
        A dictionary containing the measurement counts for each possible outcome.
    """
    tqc = transpile(circuit, backend)
    return run_counts(backend, tqc, shots)

def prob_of_zero(counts, nbits=1):
    """
    Calculate the probability of measuring all qubits in the |0> state.

    Args:
        counts: A dictionary containing the measurement counts for each possible outcome.
        nbits: The number of qubits in the circuit (default is 1).

    Returns:
        The probability of measuring all qubits in the |0> state.
    """
    key = "0" * nbits
    return counts.get(key, 0) / sum(counts.values())

def backend_summary(backend, max_rows=8, single_qubit_gates=None, two_qubit_gates=None):
    """
    Generate a summary of the backend's properties, including T1, T2, readout error, and gate errors.

    Args:
        backend: The quantum backend to summarize.
        max_rows: The maximum number of qubits to include in the summary (default is 8).
        single_qubit_gates: A list of single-qubit gate names to include in the summary (default is None).
        two_qubit_gates: A list of two-qubit gate names to include in the summary (default is None).

    Returns:
        Two pandas DataFrames: one for qubit properties and one for coupling properties.
    """
    qubit_rows = []
    props = backend.properties()
    for q in range(min(backend.num_qubits, max_rows)):
        qubit_rows.append({
            "qubit": q,
            "T1 (us)": None if props is None else round(props.t1(q) * 1e6, 2),
            "T2 (us)": None if props is None else round(props.t2(q) * 1e6, 2),
            "readout_error": None if props is None else round(props.readout_error(q), 5),
        })
        if single_qubit_gates is not None and props is not None:
            for gate in single_qubit_gates:
                error = props.gate_error(gate, [q])
                if error is not None:
                    qubit_rows[-1][f"{gate} error"] = round(error, 5)

    coupling_rows = []
    if two_qubit_gates is not None and props is not None:
        for gate in two_qubit_gates:
            for pair in backend.configuration().coupling_map:
                error = props.gate_error(gate, pair)
                if error is not None:
                    coupling_rows.append({
                        "qubit1": pair[0],
                        "qubit2": pair[1],
                        f"{gate} error": round(error, 5)
                    })

    return pd.DataFrame(qubit_rows), pd.DataFrame(coupling_rows)


<a id="part-a"></a>

# Part A — Noise, Backends, and Benchmarking

A quantum circuit is still the logical description of the computation, but the way it is
executed on **real quantum hardware** depends on calibrated backend information such as
native gates, connectivity, gate fidelities, readout performance, and coherence times.
This part treats those hardware details as practical engineering information: they help
us choose layouts, build backend-informed simulators, and compare how different
architectures affect transpiled circuits and measurement outcomes.

By the end of Part A, you should be able to answer four practical questions:

1. **How do hardware noise and circuit size affect measured results?**
2. **What information does a backend contain?**
3. **How can we simulate backend-informed noise in Qiskit?**
4. **How can we compare different backends in a simple, quantitative way?**


<a id="part-a-ch1"></a>

## Chapter 1 — Noise in quantum computers and backend-aware simulation

In the circuit model, the output of a circuit is determined by the operations we write down.
On quantum hardware, the measured result also depends on the **quality and structure of the backend**.

Once we connect a circuit to a backend, the device becomes part of the computational workflow.
Two logically equivalent circuits can behave differently if they are executed on different backends, or even on different qubits of the same backend.

There are three beginner-level ideas to keep in mind.

- **Gate errors:** every gate is imperfect, and two-qubit gates are usually noisier than one-qubit gates.
  - Generally, we quantify gate error by the the *error rate*.
- **Decoherence:** qubits gradually lose quantum information over time, often characterized by $T_1$ and $T_2$.
  - It happens for example when a qubit state decays to |0> state with age.
  - Or for exmaple when a qubit loses its superposition and become pure fluctuation.
- **Noise accumulation:** larger circuits are generally harder because they contain more gates and take more time.

Here, $T_1$ is the energy-relaxation time, which describes how quickly an excited qubit tends to relax toward $|0\rangle$. $T_2$ is the phase-coherence time, which describes how quickly relative phase information in a superposition decays.

To reason about these effects in practice, we need to read backend information and use that information in simulation.
In this chapter, we will first inspect what a backend tells us, and then use the same backend model in simple noisy experiments.

### 1.1 Looking at backend information

If you open [IBM Quantum Platform](https://quantum.cloud.ibm.com/computers) and inspect a real backend, you will typically see properties such as

- basis gates,
- coupling map (showing connectivity),
- CLOPS (circuit layer operations per second),
- one-qubit and two-qubit gate errors,
- readout errors,
- $T_1$ and $T_2$ values.

These properties give a first estimate of how to match a circuit to the backend.
For example, gate and readout errors provide useful information for choosing qubit layouts and transpilation strategies.
The $T_1$ and $T_2$ times provide characteristic relaxation and coherence timescales for selecting circuit depths and execution schedules that are well matched to the device.

This is often the first place to look before running a circuit.
You try to understand what kind of hardware you are dealing with before you decide how to transpile, on which qubits to place the circuit, and how much noise to expect.

### 1.2 What information defines a backend?

When we prepare to run a circuit on a backend, it is important to consider these questions:

- What gates does the device natively support?  
- Which qubits are directly connected?  
- How noisy are the available operations and measurements?

Together, these properties determine how the transpiler rewrites your circuit and how much error you should expect when it is executed.

In the next few cells, we inspect these ingredients one by loading a real IBM Quantum device `ibm_kingston`.

In [ ]:
# Get the IBM Quantum API key and instance from api.json.
# Make sure to set these environment variables in your api.json file.
# Example api.json:
# {
#   "IBM_QUANTUM_API_KEY": "your_api_key_here",
#   "IBM_QUANTUM_CRN": "your_instance_here"
# }

# import os
# import json

# with open("api.json") as f:
#     config = json.load(f)

# config = {
#   "IBM_QUANTUM_API_KEY": "your_api_key_here",
#   "IBM_QUANTUM_CRN": "your_instance_here"
# }

os.environ["IBM_QUANTUM_API_KEY"] = config["IBM_QUANTUM_API_KEY"]
os.environ["IBM_QUANTUM_CRN"] = config["IBM_QUANTUM_CRN"]

In [ ]:
# Get the IBM Quantum API key and instance from environment variables

token = os.getenv("IBM_QUANTUM_API_KEY")
instance = os.getenv("IBM_QUANTUM_CRN")

if not token or not instance:
    raise ValueError("IBM_QUANTUM_API_KEY or IBM_QUANTUM_CRN is missing from the .env file.")


from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token=token,
    instance=instance,
    overwrite=True,
    set_as_default=True,
)

In [ ]:
service = QiskitRuntimeService()

try:
    print("Available backends:", service.backends())
    backend_name = "ibm_kingston"  # Change this to the desired backend name
    backend = service.backend(backend_name)
except QiskitBackendNotFoundError as e:
    print(
        f"Could not find '{backend_name}' among the available Open Plan backends. "
        "The backend may be temporarily unavailable, under maintenance, or not accessible "
        "from the current account/region. Falling back to FakeNighthawk."
    )
    backend = FakeNighthawk()


print("Backend:", backend.name)
print("\nBasis operations:")
print(backend.operation_names)
print("\nCoupling-map edges:")
print(backend.coupling_map.get_edges())
display(backend.coupling_map.draw())

single_qubit_summary, two_qubit_summary = backend_summary(
    backend,
    max_rows=8,
    single_qubit_gates=["sx", "x"],
    two_qubit_gates=["cz"])

print("Qubit Properties: (8 qubits only)")
display(single_qubit_summary)

print("\nTwo-Qubit Properties: (8 couplings only)")
display(two_qubit_summary.head(8))

When a circuit is transpiled for a real backend, limited connectivity and basis operations introduce additional overheads to the resulting circuit:

- additional SWAP gates are required to entangle two qubits out of the connectivity,
- the gates should be decomposed into available ones.

Also, the table above shows that backend properties are not perfectly uniform across qubits. Gate errors, readout errors, and coherence times can vary from one qubit to another.

<div class="alert alert-block alert-success">

**Exercise 1 — Inspect a Heron backend.**

Load the 156-qubit Heron system `ibm_fez`, and submit its basis operations and coupling map.

</div>

In [ ]:
# TODO: Add your code here
# basis_operations =  # list[str]
# coupling_map =  # list[list[int] | tuple[int, int]] |

### 1.3 Create a simple noisy simulator

There are two useful ways to simulate noisy backends in Qiskit.

The first approach is to construct a noisy simulator from backend information.

The backend can be either a fake backend or a real backend that is accessible from your IBM Quantum account. In both cases, `AerSimulator.from_backend(...)` uses backend information such as basis gates, coupling map, and calibration-based noise properties to build a simulator that reflects device-level characteristics.

This is convenient when we want to study how a circuit behaves under backend-informed noise while still running the experiment as a simulation.

The second is to build a **custom noise model** ourselves.
This is useful when we want a controlled experiment.
Instead of inheriting all device properties at once, we can choose a specific type of noise and adjust its strength directly.
This makes it easier to see how a single noise mechanism changes the outcome of a circuit.

Check documentation for more information:
- [**Depolarizing Error**](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.depolarizing_error.html)
- [**Readout Error**](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.ReadoutError.html)

In [ ]:
# 1. Backend-informed noisy simulator
# This backend can be a fake backend or a real backend accessible from your account.
backend_name = "ibm_marrakesh"  # Change this to the desired backend name
backend = service.backend(backend_name)  # Loading the backend from the IBM Quantum service
backend_simulator = AerSimulator.from_backend(backend)  # Loading the properties of the backend


# 2. Custom noisy simulator defined directly by basis gates + connectivity + noise
basis_gates_single_noisy = ["sx", "x"]
basis_gates_single_ideal = ["rz", "measure"]
basis_gates_two_qubit = ["cx"]
basis_gates_custom = basis_gates_single_noisy + basis_gates_single_ideal + basis_gates_two_qubit

coupling_map_custom = [[0, 1], [1, 2], [2, 3], [3, 4]]
coupling_map_custom += [[1, 0], [2, 1], [3, 2], [4, 3]]  # add reverse directions for undirected connectivity

lambda_1q = 0.003
lambda_2q = 0.03
p_readout = 0.015

noise_model_custom = NoiseModel(basis_gates=basis_gates_custom)

# 1-qubit gate noise
err_1q = depolarizing_error(lambda_1q, num_qubits=1)
for gate in basis_gates_single_noisy:
    noise_model_custom.add_all_qubit_quantum_error(err_1q, gate)

# 2-qubit gate noise
err_2q = depolarizing_error(lambda_2q, num_qubits=2)
for gate in basis_gates_two_qubit:
    noise_model_custom.add_all_qubit_quantum_error(err_2q, gate)

# 1-qubit readout noise
readout_error_matrix = [
    [1 - p_readout, p_readout],
    [p_readout, 1 - p_readout],
]

# Ensure the readout error matrix is valid stochastic matrix (rows sum to 1)
assert np.allclose(np.sum(readout_error_matrix, axis=1), 1), "Readout error matrix rows must sum to 1."

readout_err = ReadoutError(readout_error_matrix)

noise_model_custom.add_all_qubit_readout_error(readout_err)

# Finally define the custom noisy simulator
sim_custom_backend = AerSimulator(
    noise_model=noise_model_custom,
    basis_gates=basis_gates_custom,
    coupling_map=coupling_map_custom,
)

### 1.4 First experiment: ideal versus noisy GHZ

In this first experiment, we follow the Qiskit pattern:

1. **Map** the problem to a quantum circuit by preparing a GHZ state.
2. **Optimize** the circuit for the backend or simulator using transpilation.
3. **Execute** the circuit on an ideal simulator and a backend-informed noisy simulator.
4. **Analyze** the output distributions and compare them with the ideal GHZ behavior.

To inspect how the noise affects the outcome, we now build a simple GHZ circuit and compare

- an ideal simulator,
- a fake-backend simulator with realistic hardware noise.

The GHZ circuit is a good first example because its ideal output is very simple.
In the absence of noise, the probability should be concentrated on only two bitstrings: all zeros and all ones.


#### Qiskit pattern step 1: Map
We first map the experiment to a quantum circuit by preparing a 5-qubit GHZ state and measuring all qubits.

In [ ]:
n = 5

qc_ghz = QuantumCircuit(n, n)
qc_ghz.h(0)
for i in range(n - 1):
    qc_ghz.cx(i, i + 1)
qc_ghz.barrier()
qc_ghz.measure(range(n), range(n))

qc_ghz.draw("mpl")

#### Qiskit pattern step 2: Optimize

Next, we transpile the GHZ circuit for the backend-informed simulator. This adapts the circuit to the backend's basis gates and coupling map.

In [ ]:
print(f"Transpiling the circuit for the backend {backend.name}...")
pm = generate_preset_pass_manager(backend)
transpiled_qc_ghz = pm.run(qc_ghz)

#### Qiskit pattern step 3: Execute

We now execute the circuit on a backend-informed noisy simulator.

In [ ]:
shots = 4096

print(f"Running the circuit on the {backend_simulator.name} noisy simulator with {shots} shots...")

sampler = Sampler(mode=backend_simulator)                       # Create a sampler for the noisy simulator
job = sampler.run([transpiled_qc_ghz], shots=shots)             # Run the transpiled circuit on the noisy simulator
result = job.result()                                           # Get the result of the execution
print(f"Result:\n {result}")

cr = list(result[0].data.keys())                                # Get the keys of the data
noisy_simulator_counts = result[0].data[cr[0]].get_counts()     # Get the measurement counts from the result data first key
print(f"Noisy simulator counts:\n {noisy_simulator_counts}")

#### Qiskit pattern step 4: Analyze

Finally, we compare the output distributions. The ideal simulator should mainly return `00000` and `11111`, while the backend-informed noisy simulator can show how backend characteristics spread probability to other bitstrings.

In [ ]:
ideal_backend = AerSimulator()

ideal_counts = transpile_and_run_counts(ideal_backend, qc_ghz)

print("Ideal counts (top 5):")
for bitstring, count in sorted(ideal_counts.items(), key=lambda item: -item[1])[:5]:
    print(f"  {bitstring}: {count}")

print("\nNoisy counts from backend simulator (top 5):")
for bitstring, count in sorted(noisy_simulator_counts.items(), key=lambda item: -item[1])[:5]:
    print(f"  {bitstring}: {count}")

In [ ]:
plot_histogram(
    [ideal_counts, noisy_simulator_counts],
    legend=["ideal", "noisy backend"],
    figsize=(12, 5)
)

The ideal simulator should put almost all probability on `00...0` and `11...1`.
The fake backend should still preserve those two outcomes as the most important ones, but it usually assigns some weight to additional bitstrings as well.

<a id="part-a-ch2"></a>

## Chapter 2 — What kind of noise do we care about?

Before discussing noise, it is useful to slightly generalize how we describe quantum states.

So far, a quantum state may have been written as a statevector, such as $|\psi\rangle$.
This is enough for an ideal, perfectly isolated quantum system.
However, real quantum computers are not perfectly isolated: qubits interact weakly with their environment, gates are imperfect, and measurements are noisy.
To describe these situations, we use a **density matrix** $\rho$.

For a pure state $|\psi\rangle$, the density matrix is

$$
\rho = |\psi\rangle\langle\psi|.
$$

More generally, $\rho$ can also represent a **mixed state**, where the system is in different quantum states with different probabilities.
This makes density matrices a natural language for noisy quantum systems.

A **quantum channel** is a mathematical description of a physical process acting on a density matrix:

$$
\rho \mapsto \mathcal{E}(\rho).
$$

An ideal quantum gate is a special case of a quantum channel. If the gate is represented by a unitary matrix $U$, then

$$
\rho \mapsto U\rho U^\dagger.
$$

Noise is also described by quantum channels, but generally not by a single unitary operation on the system alone. Instead, noisy channels capture effects such as random gate errors, loss of phase coherence, relaxation toward $|0\rangle$, and imperfect measurement.

<div class="alert alert-block alert-info">
For more background on density matrices, see this lesson in IBM Quantum Learning:
<a href="https://quantum.cloud.ibm.com/learning/courses/general-formulation-of-quantum-information/density-matrices/density-matrix-basics">
Density matrix basics
</a>.
</div>

In this chapter, we study three important models of noise. These are not the only noise mechanisms that matter on real hardware, but they are useful models for developing intuition. Each one behaves differently.

| Noise model | What it represents | Channel form |
|---|---|---|
| **Depolarization error** | A simple model of gate imperfection where the state is partially randomized. | $$\rho \mapsto (1-\lambda)\rho + \lambda \frac{I}{2}$$ |
| **Pauli error** | A structured stochastic error model using $X$, $Y$, and $Z$ errors. | $$\rho \mapsto p_I \rho + p_X X\rho X + p_Y Y\rho Y + p_Z Z\rho Z$$ |
| **Thermal relaxation** | A time-dependent noise model related to energy relaxation and dephasing. | $$\rho=\begin{pmatrix}\rho_{00}&\rho_{01}\\ \rho_{10}&\rho_{11}\end{pmatrix} \mapsto \begin{pmatrix}\rho_{00}+(1-e^{-t/T_1})\rho_{11}&e^{-t/T_2}\rho_{01}\\ e^{-t/T_2}\rho_{10}&e^{-t/T_1}\rho_{11}\end{pmatrix}$$ |


<div class="alert alert-block alert-info">
For more classes of quantum errors, see the
<a href="https://quantum.cloud.ibm.com/docs/guides/build-noise-models#quantum-errors">
list of quantum errors
</a>.
</div>

### 2.1 Depolarization error

Depolarization is the simplest way to say:

> after a gate, the state becomes partially randomized.

This model is useful because it captures the idea of an imperfect gate without committing to a detailed physical mechanism.  
For a single qubit, the depolarizing channel can be written schematically as
$$
\rho \mapsto (1-\lambda)\rho + \lambda \frac{I}{2},
$$
where a larger $\lambda$ means more randomization.

In order to observe the effect of depolarization, we can design a simple experiment:

- start from $|0\rangle$,
- apply an even number $2n$ of `x` gates,
- measure in the computational basis,
- observe how the survival probability $\Pr(0)$ decreases as $n$ grows.

In an ideal circuit, an even number of `x` gates returns the qubit to $|0\rangle$ exactly.
Under depolarizing noise, the state becomes partially randomized after passing each of the gates, which becomes less reliable as the sequence length $n$ grows.

Now, let's visualize the effect of the noise and measure them.

<div class="alert alert-block alert-success">

**Exercise 2 — Repeated-X circuit.**

Given `n`, complete `repeated_x_circuit(n)` so that it returns a single-qubit circuit applying $2n$ `x` gates followed by a measurement.

</div>

In [ ]:
'''
def repeated_x_circuit(n):
    qc = QuantumCircuit(1, 1)
    # TODO: Add your code here

    return qc
'''

print("Example circuit for the repeated X experiment with n=4:")
display(repeated_x_circuit(4).draw("mpl"))

In [ ]:
def depol_x_simulator(lambda_param):
    """
    Create a noisy simulator with depolarizing noise for X gates.
    """
    noise_model = NoiseModel()

    error = depolarizing_error(lambda_param, num_qubits=1)
    noise_model.add_all_qubit_quantum_error(error, ["x"])

    return AerSimulator(noise_model=noise_model)

lambdas = [0.00, 0.02, 0.04]    # List of depolarizing noise parameters to test
n_values = list(range(0, 31))   # List of n values for the repeated X experiment

shots = 20_000

fig, ax = plt.subplots(figsize=(7, 4))

p0 = {}

for lambda_param in lambdas:
    backend = depol_x_simulator(lambda_param)
    p0[lambda_param] = []
    for n in n_values:
        qc = repeated_x_circuit(n)
        counts = run_counts(backend, qc, shots=shots)
        p0[lambda_param].append(prob_of_zero(counts))
    ax.plot(n_values, p0[lambda_param], marker="o", label=f"lambda={lambda_param}")

ax.set_xlabel("$n$ in $2n$ repeated X gates")
ax.set_ylabel("Pr(0)")
ax.set_title("Depolarizing noise: survival under repeated X gates")
ax.legend()
ax.grid(True)
plt.show()

Even though an ideal sequence of `2n` `x` gates should always return the qubit to $|0\rangle$, the observed probability $\Pr(0)$ decreases as the sequence gets longer and the depolarization error becomes more severe.

This is a direct visual example of **noise accumulation**.

In theory, after each noisy `x` gate, the information is shrunk by a factor of $(1-\lambda)$. Since the circuit contains $2n$ noisy `x` gates, the survival probability becomes

$$
\Pr(0)
=
\frac{1 + (1-\lambda)^{2n}}{2}.
$$

This means that $\Pr(0)$ decays exponentially toward $1/2$, which corresponds to a completely randomized single-qubit state. Therefore, by fitting the measured survival probability to this curve, we can estimate the depolarization parameter $\lambda$.

The code below uses `curve_fit` from `scipy.optimize` to find the value of $\lambda$ that best matches the observed data. For each simulated noise strength, it compares the sampled probabilities with the fitted curve, plots both on the same graph, and summarizes the true and fitted values of $\lambda$ in a table.

Because the probabilities are estimated from a finite number of shots, the fitted value does not have to match the input value exactly. However, if the model is correct and the number of shots is large enough, the fitted $\lambda$ should be close to the value used in the simulator.

In [ ]:
def depol_survival_model(n, lambda_param):
    return 0.5 + 0.5 * (1 - lambda_param) ** (2 * n)

fit_rows = []

fig, ax = plt.subplots(figsize=(7, 4))

for lambda_true in lambdas:
    y_data = np.array(p0[lambda_true])

    popt, pcov = curve_fit(
        depol_survival_model,
        n_values,
        y_data,
        p0=[0.1],           # Initial guess for lambda
        bounds=(0.0, 1.0),  # Bounds for lambda
    )

    lambda_fit = popt[0]
    lambda_std = np.sqrt(np.diag(pcov))[0]

    fit_rows.append({
        "true lambda": lambda_true,
        "fitted lambda": lambda_fit,
        "fit std": lambda_std,
    })

    ax.plot(n_values, y_data, "o", label=f"data, lambda={lambda_true}")
    ax.plot(
        n_values,
        depol_survival_model(np.array(n_values), lambda_fit),
        "-",
        label=f"fit, lambda={lambda_fit:.4f}",
    )

ax.set_xlabel("$n$ in $2n$ repeated X gates")
ax.set_ylabel("Pr(0)")
ax.set_title("Fitting depolarizing noise from survival probability")
ax.legend()
ax.grid(True)
plt.show()

fit_df = pd.DataFrame(fit_rows)
display(fit_df)

### 2.2 Pauli error

Pauli error is more structured than depolarization.

After each gate, one of the Pauli operations, $I$, $X$, $Y$, or $Z$, is applied with a certain probability:

$$
\rho \mapsto p_I \rho + p_X X\rho X + p_Y Y\rho Y + p_Z Z\rho Z,
$$

where
$
p_I = 1 - p_X - p_Y - p_Z.
$

If the non-identity error probabilities are uniform,
$$
p_X = p_Y = p_Z = \frac{\lambda}{4},
$$
then this Pauli error model becomes identical to the depolarization model stated above.

The key difference is that Pauli error allows the noise to be stronger along one direction than another. In other words, the error does not have to randomize the state equally in all directions.

For example, we can identify two important types of Pauli errors:

- **Bit-flip-like errors**, caused by $X$ and $Y$ errors, which can change computational-basis states such as $|0\rangle$ and $|1\rangle$.
- **Phase-flip-like errors**, caused by $Z$ and $Y$ errors, which can change the relative phase between basis states.

The $Y$ error appears in both categories because, up to a global phase,

$$
Y = iXZ.
$$

Therefore, an $Y$ error acts like a combination of a bit flip and a phase flip.

This is why a single experiment is not enough to tell the whole story. A bit-flip-like error can be easy to see in one basis and almost invisible in another. Likewise, a phase-flip-like error can be completely hidden if we only prepare computational-basis states and measure in the computational basis.

We therefore use **two complementary experiments**.

#### Experiment A: bit-flip-sensitive test

We use the same repeated-`x` experiment as in the depolarization case.

The difference is in how we interpret the result. Under Pauli noise, the survival probability $\Pr(0)$ mainly reveals the bit-flip-like part of the error channel.

Errors with an $X$ component can change $|0\rangle$ into $|1\rangle$, so they are directly visible in computational-basis measurement. A $Y$ error also contributes to this behavior because it contains a bit-flip component. In contrast, a pure $Z$ error only changes the phase and is therefore not visible when we only measure in the computational basis.

Therefore, this experiment is useful for detecting $X$- and $Y$-type errors, but it is not sensitive to pure $Z$-type phase errors.

#### Experiment B: phase-flip-sensitive test

We now use a complementary version of the repeated-`x` experiment.

Instead of starting from $|0\rangle$, we start from the $X$-basis state $|+\rangle$ by applying a Hadamard gate:

$$
|0\rangle \mapsto |+\rangle.
$$

Then we apply the same repeated `x` gates and finally measure in the $X$ basis. In the circuit, measuring in the $X$ basis can be implemented by applying another Hadamard gate before the computational-basis measurement.

The survival probability $\Pr(0)$ now represents the probability that the final state is still $|+\rangle$.

This experiment is sensitive to phase-flip-like errors. A pure $Z$ error changes

$$
|+\rangle \mapsto |-\rangle,
$$

so it becomes directly visible in the $X$-basis measurement. A $Y$ error also contributes because it contains a phase-flip component. In contrast, a pure $X$ error does not change $|+\rangle$ except for an eigenvalue phase, since

$$
X|+\rangle = |+\rangle.
$$

Therefore, this experiment is useful for detecting $Z$- and $Y$-type errors, but it is not sensitive to pure $X$-type bit-flip errors.

<div class="alert alert-block alert-success">

**Exercise 3 — Repeated-X circuit measured in the X basis.**

Given `n`, complete `repeated_x_meas_x_circuit(n)` so that it returns a single-qubit circuit applying $2n$ `x` gates to the $|+\rangle$ state and measuring in the X basis.

**Hint:** apply Hadamard gates before and after the $2n$ `x` gates.

</div>

In [ ]:
def repeated_x_meas_x_circuit(n):
    qc = QuantumCircuit(1, 1)
    # TODO: Add your code here

    return qc

In [ ]:
n = 5

print(f"Circuit for Experiment A (bit-flip-sensitive test) with n={n}:")
display(repeated_x_circuit(n).draw("mpl"))

print(f"\nCircuit for Experiment B (phase-flip-sensitive test) with n={n}:")
display(repeated_x_meas_x_circuit(n).draw("mpl"))

In [ ]:
pauli_cases = {
    "Z only":       {"px": 0.00, "py": 0.00, "pz": 0.02},
    "X only":       {"px": 0.02, "py": 0.00, "pz": 0.00},
    "X + Y":        {"px": 0.02, "py": 0.02, "pz": 0.00},
    "X + Y + Z":    {"px": 0.02, "py": 0.02, "pz": 0.02},
}

n_values = list(range(0, 31))

fig, axs = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for label, probs in pauli_cases.items():
    px, py, pz = probs["px"], probs["py"], probs["pz"]

    # Build the Pauli noise model
    noise_model_pauli = NoiseModel()
    noise_model_pauli.add_all_qubit_quantum_error(
        pauli_error([
            ("I", 1 - px - py - pz),
            ("X", px),
            ("Y", py),
            ("Z", pz),
        ]),
        ["x"],
    )
    backend_pauli = AerSimulator(noise_model=noise_model_pauli)

    p0_z_basis = []
    p0_x_basis = []

    for n in n_values:
        # Experiment A: Z-basis survival test
        qc_z = repeated_x_circuit(n)
        counts_z = run_counts(backend_pauli, qc_z, shots=shots)
        p0_z_basis.append(prob_of_zero(counts_z))

        # Experiment B: X-basis survival test
        qc_x = repeated_x_meas_x_circuit(n)
        counts_x = run_counts(backend_pauli, qc_x, shots=shots)
        p0_x_basis.append(prob_of_zero(counts_x))

    axs[0].plot(n_values, p0_z_basis, marker="o", label=label)
    axs[1].plot(n_values, p0_x_basis, marker="o", label=label)

axs[0].set_title("Experiment A: Z-basis survival test")
axs[0].set_xlabel("n in 2n repeated X gates")
axs[0].set_ylabel("Survival probability")
axs[0].grid(True)

axs[1].set_title("Experiment B: X-basis survival test")
axs[1].set_xlabel("n in 2n repeated X gates")
axs[1].grid(True)

axs[0].legend()
# axs[1].legend()

plt.tight_layout()
plt.show()

The two panels show the same Pauli error cases viewed in two different bases.

In the $Z$-basis test, the survival probability mainly decreases when the noise contains an $X$ or $Y$ component. A pure $Z$ error barely changes the result, because its effect is only a phase change in the computational basis.

The opposite pattern appears in the $X$-basis test. There, a pure $X$ error is almost invisible, while $Z$ and $Y$ errors reduce the survival probability because they move $|+\rangle$ toward $|-\rangle$.

<div class="alert alert-block alert-success">

**Exercise 4 — Which Pauli errors are visible?**

For each Pauli error, identify whether it can change the final measurement outcome in the Z-basis (bit-flip-sensitive) and X-basis (phase-flip-sensitive) repeated-X experiments.

</div>

In [ ]:
# TODO: Replace every None with either True or False.
# Does the error type affect the outcome of the experiment? If yes then True, otherwise False.
answer_pauli_visibility = {
    "X error": {"bit-flip sensitive": None, "phase-flip sensitive": None},
    "Y error": {"bit-flip sensitive": None, "phase-flip sensitive": None},
    "Z error": {"bit-flip sensitive": None, "phase-flip sensitive": None},
}

print("Submitted answer")
for error, contents in answer_pauli_visibility.items():
    for experiment, visibility in contents.items():
        if visibility is None:
            print(f"\t{error}: no answer submitted for the {experiment} experiment.")
        elif visibility:
            print(f"\t{error} can be observed in the {experiment} experiment.")
        else:
            print(f"\t{error} cannot be observed in the {experiment} experiment.")

### 2.3 Thermal relaxation

Thermal relaxation is different from the other two errors, where the noise applies **per gate**, but rather

> **noise per unit time.**

Since time itself becomes a source of error, even if two circuits perform the same logical operation, their performance can differ if one takes longer to execute.

In the previous models, we mainly asked what happens when an imperfect gate is applied. In thermal relaxation, we ask what happens simply because time passes while the qubit is not perfectly isolated from its environment.

Two standard parameters describe this effect:

- $T_1$: energy relaxation time,
- $T_2$: phase coherence time (**dephasing**).

Roughly speaking, $T_1$ describes how quickly an excited state $|1\rangle$ decays toward the ground state $|0\rangle$. Meanwhile, $T_2$ describes how quickly phase coherence is lost between $|0\rangle$ and $|1\rangle$.

For a single-qubit density matrix,

$$
\rho =
\begin{pmatrix}
\rho_{00} & \rho_{01} \\
\rho_{10} & \rho_{11}
\end{pmatrix},
$$

a simplified thermal relaxation model can be written as

$$
\rho(t)=
\begin{pmatrix}
\rho_{00} + (1-e^{-t/T_1})\rho_{11} & e^{-t/T_2}\rho_{01} \\
e^{-t/T_2}\rho_{10} & e^{-t/T_1}\rho_{11}
\end{pmatrix}.
$$

This expression shows two different effects. The excited-state population $\rho_{11}$ shrinks over time due to $T_1$ relaxation, while the off-diagonal coherence terms $\rho_{01}$ and $\rho_{10}$ shrink due to $T_2$ dephasing.

**Delay experiments** are useful for studying thermal relaxation. By inserting idle time into a circuit, we can separate the effect of elapsed time from the effect of applying many imperfect gates.

#### Experiment A: $T_1$-type decay

We prepare $|1\rangle$, wait for a controllable delay time, and measure in the computational basis.

This experiment mainly tracks population decay. If the delay becomes longer, the excited state is increasingly likely to relax toward $|0\rangle$.

Documentation : [Thermal Relaxation Error](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.thermal_relaxation_error.html)

In [ ]:
def t1_delay_circuit(delay_dt):
    """
    Create a quantum circuit that prepares the |1> state, applies a delay, and measures the qubit.

    Args:
        delay_dt: The duration of the delay in nanoseconds.
    Returns:
        A QuantumCircuit object representing the T1 delay experiment.
    """
    qc = QuantumCircuit(1, 1)
    qc.x(0)
    qc.delay(delay_dt, 0, unit="ns")
    qc.measure(0, 0)
    return qc

t1_us = 60
t2_us = 40

delay_values_ns = np.linspace(0, 120_000, 13)  # 0 to 120 us
p1_vals = []

for delay_ns in delay_values_ns:
    noise_model = NoiseModel()
    err = thermal_relaxation_error(
        t1=t1_us * 1e3,
        t2=t2_us * 1e3,
        time=delay_ns)
    noise_model.add_all_qubit_quantum_error(err, ["delay"])
    backend = AerSimulator(noise_model=noise_model)

    qc = t1_delay_circuit(int(delay_ns))
    counts = transpile_and_run_counts(backend, qc, shots=4096)
    p1 = counts.get("1", 0) / sum(counts.values())
    p1_vals.append(p1)

plt.figure(figsize=(7, 4))
plt.plot(delay_values_ns / 1000, p1_vals, marker="o")   # in microseconds
plt.xlabel("delay time (\u03bcs)")
plt.ylabel("Pr(1)")
plt.title("Thermal relaxation: T1-type population decay")
plt.grid(True)
plt.show()

#### Experiment B: $T_2$-type coherence decay

Now we prepare $|+\rangle$, insert a delay, and measure in the $X$ basis.

This experiment tracks how phase coherence is lost over time. Even when population has not fully relaxed, coherence can still decay, which means superposition-based behavior becomes harder to preserve.

In [ ]:
def t2_delay_circuit(delay_dt):
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.delay(delay_dt, 0, unit="ns")
    qc.h(0)
    qc.measure(0, 0)
    return qc

pplus_vals = []

for delay_ns in delay_values_ns:
    noise_model = NoiseModel()
    err = thermal_relaxation_error(t1_us * 1e3, t2_us * 1e3, time=delay_ns)
    noise_model.add_all_qubit_quantum_error(err, ["delay"])
    backend = AerSimulator(noise_model=noise_model)

    qc = t2_delay_circuit(int(delay_ns))
    counts = transpile_and_run_counts(backend, qc, shots=4096)
    pplus_vals.append(prob_of_zero(counts))

plt.figure(figsize=(7, 4))
plt.plot(delay_values_ns / 1000, pplus_vals, marker="o")
plt.xlabel("delay time (\u03bcs)")
plt.ylabel("Pr(+)")
plt.title("Thermal relaxation: T2-type coherence decay")
plt.grid(True)
plt.show()

### 2.4 Quick summary of the three toy models

- **Depolarization:** isotropic gate imperfection.
- **Pauli error:** directional stochastic $X/Y/Z$ error.
- **Thermal relaxation:** time-dependent decay governed by $T_1$ and $T_2$.

These models are simplified, but they help us interpret what we observe on more realistic backends. The important point is not to memorize the labels, but to connect each model to a physical intuition: randomization, directionality, and time dependence.

<a id="part-a-ch3"></a>

## Chapter 3 — Comparing Heron and Nighthawk Architectures

Real quantum processors exhibit diverse **architectures**, each characterized by unique *connectivity* patterns and *performance* characteristics that fundamentally impact quantum algorithm execution. This section examines two IBM Quantum architectures: the Heron Architecture, which employs a Heavy-Hex topology featuring a hexagonal lattice pattern optimized for error correction, and the Nighthawk Architecture, which offers alternative connectivity optimized for specific quantum operations. Understanding these architectural differences is crucial because the physical layout of qubits directly determines which qubits can interact, the number of SWAP gates required (and thus circuit overhead), overall circuit depth and fidelity, and ultimately, algorithm performance on real hardware.

### 3.1 Layout Comparison

The layout in a QPU is the mapping of logical qubits to physical qubits in real quantum hardware. This concept is fundamental to understanding how quantum circuits are executed on real processors, as it establishes the correspondence between the abstract qubits we define in our algorithm and the physical qubits that exist in the quantum device.

Logical qubits are the abstract units we use when designing a quantum circuit, typically represented as q[0], q[1], q[2], etc. On the other hand, physical qubits are the real elements of quantum hardware, each with a specific physical position and particular characteristics such as error rates and coherence times. The distinction between these two types of qubits is crucial because quantum hardware has physical limitations that do not exist in the abstract model.

When constructing a quantum circuit, physical qubits are assigned by default to their corresponding logical qubits, as demonstrated in this example.

In [ ]:
# virtual layout
layout = {idx:value for idx,value in enumerate(qc_ghz.qregs[0])}

layout

In [ ]:
qc_ghz.draw("mpl")

Through transpilation, it is possible to specify which physical hardware qubits should be assigned to logical qubits, though this depends on the hardware configuration. A specific example is demonstrated below using the initial_layout attribute.

In [ ]:
qc_ghz_init_1 = transpile(qc_ghz, initial_layout=[3,0,4,2,1])

qc_ghz_init_1.draw('mpl')

QPUs have specific topologies that determine which pairs of qubits can interact directly. For example, the 'Heron' architecture uses a heavy-hex (heavy hexagonal) topology, while the 'Nighthawk' architecture features a different connectivity pattern optimized for specific quantum operations. Other processors may have chain, grid, or custom topologies. In these topologies, only certain pairs of physical qubits are connected and can execute two-qubit gates such as CNOT or CZ directly. This limited connectivity is a physical constraint of the hardware that must be considered when mapping logical qubits to physical qubits. The choice between architectures like Heron and Nighthawk can significantly impact circuit performance, as their distinct connectivity patterns may favor different types of quantum algorithms or problem structures.

In [ ]:
backend_heron = FakeTorino()  # heron architecture
backend_nighthawk = FakeNighthawk()  #Nighthwak architecture

Visualize the topologies of both backends

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heavy-hex layout
plot_gate_map(backend_heron, ax=axes[0])
axes[0].set_title(f"Heavy-hex Layout\n({backend_heron.num_qubits} qubits)", fontsize=14, fontweight='bold')

# Nighthawk layout
plot_gate_map(backend_nighthawk, ax=axes[1])
axes[1].set_title(f"Nighthawk Layout\n({backend_nighthawk.num_qubits} qubits)", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

coupling_map_heron = backend_heron.coupling_map
coupling_map_night = backend_nighthawk.coupling_map

print(f"\nConnectivity Statistics:")
print(f"Heavy-hex: {len(coupling_map_heron.get_edges())} edges")
print(f"Nighthawk: {len(coupling_map_night.get_edges())} edges")
print(f"Average degree (Heavy-hex): {2 * len(coupling_map_heron.get_edges()) / backend_heron.num_qubits:.2f}")
print(f"Average degree (Nighthawk): {2 * len(coupling_map_night.get_edges()) / backend_nighthawk.num_qubits:.2f}")

The importance of layout lies in its direct impact on the quality and feasibility of quantum circuit execution. A poor layout occurs when logical qubits that need to interact in the circuit are mapped to physical qubits that are not directly connected in the hardware. In these cases, the compiler must insert additional SWAP operations to move quantum states between qubits, allowing the desired interaction to finally occur. These SWAP operations have multiple negative consequences: they increase circuit depth (the number of sequential gate layers), increase the total number of gates that must be executed, accumulate more error due to the imperfections of each additional gate, and reduce the fidelity of the final result due to quantum decoherence that occurs during the extra execution time.

Conversely, a good layout minimizes or eliminates the need for SWAP operations by intelligently assigning logical qubits that interact frequently to physical qubits that are directly connected in the hardware topology. This optimal mapping preserves the structure of the original circuit as much as possible, reducing transpilation overhead and maximizing the probability of obtaining accurate results. In variational algorithms such as QAOA, where circuit depth and gate fidelity are critical to performance, the difference between an optimal layout and a suboptimal one can determine the success or failure of the entire quantum experiment.

In [ ]:
num_qubits = 120
qc_ghz_120 = QuantumCircuit(num_qubits,num_qubits)
qc_ghz_120.h(0)

for idx in range(num_qubits-1):
    qc_ghz_120.cx(idx,idx+1)

qc_ghz_120.measure(range(num_qubits),range(num_qubits))
qc_ghz_120.draw('mpl',fold=-1)

<div class="alert alert-block alert-success">

**Exercise 5a — Initial layout and basis gates.**

Create an `initial_layout` that goes in ascending order from `q0` up to the number of qubits in `qc_ghz_120`, and restrict the circuit's `basis_gates` to `h`, `cx`, and `swap`.

</div>

In [ ]:
initial_layout = [] # TODO: Add your code here
basis_gates = [] # TODO: Add your code here

# Transpile for Heavy-hex backend
print("Transpiling GHZ circuit for Heavy-hex architecture...")
transpiled_heron = transpile(
    qc_ghz_120,
    backend=backend_heron,
    optimization_level=3,
    initial_layout = initial_layout,
    seed_transpiler=20,
    basis_gates= basis_gates

)

# Transpile for Nighthawk backend
print("Transpiling GHZ circuit for Nighthawk architecture...")
transpiled_nightawk = transpile(
    qc_ghz_120,
    backend=backend_nighthawk,
    optimization_level=3,
    initial_layout = initial_layout,
    seed_transpiler=20,
    basis_gates=basis_gates
)

<div class="alert alert-block alert-success">

**Exercise 5b — Count SWAP gates.**

Complete the `count_swap_gates` function so that it returns how many SWAP gates are used in a circuit.

</div>

In [ ]:
def count_swap_gates(circuit: QuantumCircuit) -> int:
    """
    Count the number of SWAP gates in a circuit.

    Args:
        circuit: The QuantumCircuit to analyze.

    Returns:
        The number of SWAP gates in the circuit.

    Hint:
        - Iterate over circuit.data to access each instruction
        - Each instruction has an .operation.name attribute
        - Check if the name equals "swap"
    """
    swap_count = 0
    # TODO: Add your code here

    return swap_count


# Print comparison
print("GHZ Circuit Transpilation Comparison")
print(f"\n{'Metric':<30} {'Heavy-hex':<15} {'Nighthawk':<15}")
print(f"{'Circuit Depth':<30} {transpiled_heron.depth():<15} {transpiled_nightawk.depth():<15}")
print(f"{'Total Gates':<30} {transpiled_heron.size():<15} {transpiled_nightawk.size():<15}")
print(f"{'Two-qubit Gates':<30} {transpiled_heron.num_nonlocal_gates():<15} {transpiled_nightawk.num_nonlocal_gates():<15}")
print(f"{'SWAP Gates':<30} {count_swap_gates(transpiled_heron):<15} {count_swap_gates(transpiled_nightawk):<15}")



### 3.2 Benchmarking the Previous IBM QPUs




In [ ]:
def backend_summary_detail(backend):
    conf = backend.configuration()
    props = backend.properties()
    n_qubits = backend.num_qubits

    # T1/T2/readout from BackendProperties.
    t1s, t2s, readout_errs = [], [], []
    for q in range(n_qubits):
        t1s.append(props.t1(q))
        t2s.append(props.t2(q))
        readout_errs.append(props.readout_error(q))

    gate_errs = dict()
    for gate in conf.gates:
        gate = gate.to_dict()
        if (gate["name"] in ["reset", "measure", "rz"]): # "rz" has no error
            continue
        for applied_qubits in gate["coupling_map"]:
            applied_qubits = tuple(applied_qubits)
            if applied_qubits not in gate_errs:
                gate_errs[applied_qubits] = list()
            gate_errs[applied_qubits].append(props.gate_error(gate["name"], applied_qubits))

    twoq_errs, oneq_errs = list(), list()
    for qubits, err in gate_errs.items():
        if len(qubits) == 1:
            oneq_errs.append(err)
        elif len(qubits) == 2:
            twoq_errs.append(err)
        else:
            raise Exception

    if n_qubits < 10:
        size_family = "small: <10 qubits"
    elif n_qubits < 120:
        size_family = "medium: <120 qubits"
    else:
        size_family = "large: >=120 qubits"

    return {
        "name": backend.name,
        "date": backend.online_date,
        "n_qubits": n_qubits,
        "n_coupling_edges": len(backend.coupling_map.get_edges())//2,
        "t1_median_us": np.median(t1s) * 1e6,
        "t2_median_us": np.median(t2s) * 1e6,
        "readout_error_median": np.median(readout_errs),
        "oneq_error_median": np.median(oneq_errs),
        "twoq_error_median": np.median(twoq_errs),
        "size_family": size_family
    }


In [ ]:
provider = FakeProviderForBackendV2()
backends = provider.backends()

rows = []
for backend in backends:
    if backend.name == "fake_armonk":  # single-qubit processor
        continue
    rows.append(backend_summary_detail(backend))

df = pd.DataFrame(rows)

# Remove pure test backend if desired.
# df = df[df["name"] != "fake_fractional_backend"]

df = df.sort_values(["date", "name"], na_position="last").reset_index(drop=True)

display_cols = [
    "date",
    "name",
    "n_qubits",
    "size_family",
    "t1_median_us",
    "t2_median_us",
    "oneq_error_median",
    "twoq_error_median",
    "readout_error_median",
    "n_coupling_edges",
]

display(df[display_cols].head(3))
print("...")
display(df[display_cols].tail(3))

In [ ]:
family_order = [
    "small: <10 qubits",
    "medium: <120 qubits",
    "large: >=120 qubits",
]

styles = {
    "small: <10 qubits":  {"color": "#999999", "marker": "o", "linestyle": "-"},
    "medium: <120 qubits": {"color": "#0072B2", "marker": "s", "linestyle": "--"},
    "large: >=120 qubits":  {"color": "#D55E00", "marker": "^", "linestyle": ":"},
}

metrics = [
    ("n_qubits", "Qubit count", False),
    ("n_coupling_edges", "Number of coupling edges", False),
    ("oneq_error_median", "Median 1Q gate error", True),
    ("twoq_error_median", "Median 2Q gate error", True),
    ("t1_median_us", "Median T1 [us]", False),
    ("t2_median_us", "Median T2 [us]", False),
    ("readout_error_median", "Median readout error", True),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, (col, title, logy) in zip(axes, metrics):
    for family in family_order:
        sub = df[df["size_family"] == family].sort_values("date")
        ax.scatter(
            sub["date"],
            sub[col],
            label=family,
            s=45,
            alpha=0.85,
            **(styles[family])
        )

    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.grid(alpha=0.3)

    if logy:
        ax.set_yscale("log")

# Hide the last unused subplot
axes[len(metrics)].axis("off")

# Put legend only once
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.02))

# Rotate x tick labels
for ax in axes[:len(metrics)]:
    ax.tick_params(axis="x", rotation=35)

plt.tight_layout(rect=[0, 0, 1, 0.95])

plt.show()

The last plot summarizes the evolution of IBM Quantum hardware across several fake-provider backend snapshots.

Each point corresponds to one backend snapshot.
The x-axis shows the backend's `online_date`, and the color indicates the backend size family.
Across the snapshots, we can see a clear progression from small devices with only a few qubits and coupling edges to larger processors with more than 100 qubits and substantially richer connectivity.
At the same time, the calibration-related panels show the quality metrics that matter for circuit execution: one-qubit and two-qubit gate errors, readout errors, and the $T_1$ and $T_2$ coherence times.

Overall, the plot illustrates that IBM QPUs have not only scaled in size, but have also evolved in the hardware characteristics relevant to running quantum circuits.
Later backend snapshots generally provide larger qubit counts, denser coupling maps, lower gate and readout errors, and longer coherence times.

<a id="part-b"></a>

# Part B — Error Mitigation in Qiskit Runtime

Part A showed how noise shows up in measured results. Part B turns to what we can do about
it. Chapter 1 surveys the four families of techniques for handling noise. Chapter 2 covers
the error suppression and mitigation methods available in `qiskit_ibm_runtime 0.50`.


<a id="part-b-ch1"></a>

## Chapter 1 — Errors on today's quantum hardware

Today's quantum hardware is noisy. Every gate, every idle moment, every measurement introduces small errors, and at utility scale (circuits on 100+ qubits) those errors accumulate to a degree that can no longer be ignored. Until full fault tolerance arrives, the only way to get *useful* answers out of these devices is to manage that noise, to suppress it where we can, to characterize it where we can't, and to mitigate its bias when we read out a result. Dealing with noise is therefore a central part of getting useful results from today's devices, and the techniques for doing it are an active area of development.

The techniques for handling noise fall into roughly four categories. They differ in *when* they act, in *what* they cost, and in whether they fix errors, flag them, or merely correct the statistics after the fact — so it is worth seeing all four before looking at what Qiskit Runtime offers.

- **Error correction (EC)** encodes logical information redundantly, repeatedly extracts syndromes, and when implemented fault-tolerantly, can suppress logical errors below physical error rates. Full QEC is prohibitive on near-term devices because of its qubit and time overheads.

- **Error detection (ED)** flags runs that are likely corrupted and discards them via post-selection, keeping a smaller but cleaner sample. The flag can come from a detection-only QEC code, one whose distance heralds an error without giving enough information to identify and reverse it. In that case, making ED a limiting case of error correction. ED lets a higher-level code in a hierarchical scheme treat the flagged error as an *erasure* (a known location is far easier to correct than an unknown one). It can equally come from cheaper, code-free checks, such as verifying a symmetry the ideal circuit should preserve, e.g. a conserved parity or particle number, and rejecting shots that violate it. Either way, ED trades a reduced usable sample against an implementation overhead well below that of fault-tolerant EC.

- **Error mitigation (EM)** does not fix individual shots at all. It biases or re-weights the statistics across many noisy runs so that the *estimate* of an expectation value (or distribution) is closer to the ideal one ([Cai et al., *Quantum error mitigation*, Rev. Mod. Phys. 95, 2023](https://journals.aps.org/rmp/abstract/10.1103/RevModPhys.95.045005)). This is the regime where today's utility-scale experiments operate, and it is the focus of this part.

- **Error suppression (ES)** acts during execution to reshape the noise itself. Examples include dynamical decoupling (DD), which inserts pulses on idle qubits to cancel coherent error, and Pauli twirling (PT), which randomizes gate dressings so that the residual error becomes a stochastic Pauli channel ([Wallman & Emerson, PRA 94, 2016](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325)).

Two of these four you have already used, whether you noticed or not, if you have run the Qiskit Runtime primitives. Error suppression and error mitigation are what `Estimator` implements (error correction and detection are not something the primitive applies for you). At its default option, `Estimator` already corrects for systematic readout bias (Twirled Readout Error eXtinction, TREX). You can also go further and enable additional error suppression and error mitigation methods yourself using `EstimatorOptions` as we shall see in section 2.1.2. What all of these options have in common is that they act on your circuit as a single _circuit-wide_ policy, which adds a fixed kind of overhead to every run.

Tools such as [Samplomatic](https://github.com/Qiskit/samplomatic) can be used alongside Qiskit Runtime to go beyond that _circuit-wide_ setting, managing noise layer by layer and observable by observable. In Chapter 2, we begin by giving an overview of what the familiar `EstimatorOptions` do and what they cost.

<a id="part-b-ch2"></a>

## Chapter 2 — Error mitigation as a Runtime product, and the theory behind its cost

This chapter takes the familiar Runtime options and opens them up:
- Section 2.1 walks through the error suppression and error mitigation methods `Estimator` offers and has you configure each one by hand.

- Section 2.2 then steps one level *below* the switches to the small piece of algebra — *dressing* and *commutation* — that is the most basic principle these techniques rest on.

- Section 2.3 closes the chapter by asking what these whole-circuit switches *cannot* do, and what they cost in terms of resource overhead. This motivates finer, box-by-box mitigation tools.

### 2.1  Error mitigation as a cloud service

The introduction noted that Qiskit Runtime exposes error mitigation as a product: a set of options you turn on, applied to your circuit as a single circuit-wide policy. This section catalogs those options on the `EstimatorV2` primitive — what each one does, and which `resilience_level` preset turns it on (section 2.1.1) — and then looks at each method's settings and behavior in detail (section 2.1.2). It closes with Exercise 6, in which you configure each of the five methods on its own `EstimatorOptions` so the effect of each one stays separable. The algebra these methods rest on comes in section 2.2.

#### 2.1.1  The options on the Runtime primitives

The `EstimatorV2` primitive bundles these error suppression and error mitigation techniques behind a single `EstimatorOptions` object. The official catalogue is [Error mitigation and suppression techniques](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques). The `resilience_level` option is a convenience preset that turns a fixed set of them on at once. The table below shows each technique, its role, and which preset level enables it. We will introduce the exact attribute paths in section 2.1.2.

| Technique | Role | One-line description | Enabled by |
|---|---|---|---|
| **DD** (Dynamical decoupling) | Suppression | Insert pulse sequences on idle qubits to cancel coherent errors that build up while qubits sit idle | Enable explicitly |
| **PT** (Pauli twirling, gates) | Suppression | Tailor the coherent part of 2Q-gate noise into an effectively stochastic Pauli channel, which also lets downstream methods (PEA, PEC) assume Pauli-channel noise | Level 2 |
| **TREX** (Twirled readout error extinction) | Mitigation | Twirl measurements with random X gates so the readout-error transfer matrix becomes diagonal and can be inverted by rescaling | Level 1 (default) |
| **ZNE** (Zero-noise extrapolation) | Mitigation | Amplify noise at several controlled factors (e.g. by gate folding) and extrapolate the expectation value back to the zero-noise limit (not guaranteed unbiased) | Level 2 |
| **PEA** (Probabilistic error amplification) | Mitigation | Replace ZNE's gate-folding amplification with one driven by a *learned* Pauli-Lindblad noise model, for more accurate noise scaling (still using ZNE's extrapolation step) | Set explicitly (with ZNE) |
| **PEC** (Probabilistic error cancellation) | Mitigation | Sample anti-noise circuits whose ensemble average inverts the learned noise channel, giving an unbiased estimate at the cost of a sampling overhead $\gamma^2$ | Set explicitly |

By default Runtime runs at **resilience level=1**, which enables TREX only; see the [Configure error mitigation](https://docs.quantum.ibm.com/guides/configure-error-mitigation) guide for what each level turns on. The next section looks at each technique's options and behavior in detail.


#### 2.1.2  The methods in detail

This section expands each technique from the table in section 2.1.1: what it does, the options that control it, and when it helps. Figures and full explanations are from the official [Error mitigation and suppression techniques](https://docs.quantum.ibm.com/guides/error-mitigation-and-suppression-techniques) guide.

##### Dynamical decoupling (DD) — *suppression*
Idle qubits pick up coherent errors while they wait for other qubits to finish. DD inserts a sequence of pulses on those idle qubits that multiplies out to the identity, so the intended computation is unchanged, while the pulses average away the slow coherent drift. It helps most when a circuit has idle gaps; on a densely packed circuit it can do little, or even hurt, because the pulses are themselves imperfect.

![Dynamical decoupling: an XX pulse sequence inserted during a qubit's idle period (IBM Quantum docs).](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/dd.avif)

- `dynamical_decoupling.enable` — turn DD on (off by default).
- `dynamical_decoupling.sequence_type` — which pulse sequence; default `"XX"`, others include `"XpXm"`, `"XY4"`.

##### Pauli twirling (PT) — *suppression*
Twirling (randomized compiling) sandwiches a chosen gate between random single-qubit Paulis picked so the ideal operation is unchanged. Averaged over many random instances, an arbitrary noise channel becomes a stochastic *Pauli* channel: coherent error, which otherwise accumulates quadratically with depth, is reshaped into Pauli noise that grows only linearly. Since most hardware error comes from two-qubit gates, twirling is usually applied to the native 2Q gates, and it is a prerequisite for PEA and PEC, which assume Pauli-channel noise.

![Pauli twirling replaces one circuit with a random ensemble of equivalent twirled circuits (IBM Quantum docs).](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

- `twirling.enable_gates` — twirl the 2Q gate layers.
- `twirling.num_randomizations` — number of random twirled instances.
- `twirling.shots_per_randomization` — shots per instance.

##### Twirled readout error extinction (TREX) — *mitigation*
TREX targets *measurement* error. It twirls the readout by randomly inserting an X before a measurement and flipping the classical bit back afterward. This, with readout error present, diagonalizes the readout-error transfer matrix so it can be inverted by a simple rescaling. The rescaling factor is learned by benchmarking random circuits prepared in the zero state, at the cost of a few calibration circuits. This is the one method on by default (_resilience level 1_).

![Measurement twirling: an X gate before the measurement and a classical bit-flip after are equivalent to a plain measurement when noiseless (IBM Quantum docs).](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/measurement_twirling.svg)

- `resilience.measure_mitigation` — turn TREX on.
- `resilience.measure_noise_learning.num_randomizations`, `...shots_per_randomization` — control measurement-noise learning.

##### Zero-noise extrapolation (ZNE) — *mitigation*
ZNE runs the circuit at several *amplified* noise levels and extrapolates the expectation value back to zero noise. Runtime amplifies noise by digital gate folding. This replaces a unitary $U$ with $U U^\dagger U$ (and longer foldings) to multiply its noise by a known factor and then fits the results with a chosen functional form. It often improves accuracy but is *not* guaranteed unbiased. The default samples three noise factors, for roughly a 3× overhead.

![Zero-noise extrapolation: digital gate folding amplifies noise; results are extrapolated to the zero-noise limit (IBM Quantum docs).](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/zne.avif)

- `resilience.zne_mitigation` — turn ZNE on.
- `resilience.zne.noise_factors` — amplification factors (default `(1, 3, 5)`).
- `resilience.zne.extrapolator` — fit form (e.g. linear, exponential).

##### Probabilistic error amplification (PEA) — *mitigation*
PEA is a more accurate amplifier *for* ZNE. Instead of gate folding, it first **learns** the twirled Pauli-Lindblad noise model of each entangling layer, then amplifies by probabilistically injecting single-qubit noise drawn from that learned model — a faithful scaling of the actual device noise rather than a synthetic one. The amplified results still feed ZNE's extrapolation, so PEA is switched on top of ZNE. For utility-scale experiments it is often the best amplifier choice.

- `resilience.zne_mitigation = True` together with `resilience.zne.amplifier = "pea"`.

##### Probabilistic error cancellation (PEC) — *mitigation*
PEC also learns the noise model, but instead of amplifying it, it *inverts* it: it samples an ensemble of "anti-noise" circuits whose average implements the inverse noise channel, giving an **unbiased** estimate. The price is a sampling overhead that scales as $\gamma^2$ and rises quickly with circuit depth and total noise — which is why PEC is practical only for circuits of modest size (we return to this cost in section 2.3). PEC can also be applied layer by layer with _Samplomatic_, and [shaded lightcones (SLC)](https://qiskit.github.io/qiskit-addon-slc/) is an evolution of it.

- `resilience.pec_mitigation` — turn PEC on.
- `resilience.pec.max_overhead` — cap on the sampling overhead $\gamma^2$; Runtime backs off if the cap would be exceeded.

__Note__: ZNE/PEA and PEC are *opposing* strategies — amplify the noise versus cancel it — and cannot be enabled on the **same** `EstimatorOptions`.

<div class="alert alert-block alert-success">

**Exercise 6 — Configure each method.**

Using the methods and attribute paths from section 2.1.2, configure each of the five on its own `EstimatorOptions` object, then collect them into a dictionary `options_dict` with keys `"dd"`, `"pt"`, `"trex"`, `"zne"`, `"pec"`.

- **`dd_options`** — Dynamical decoupling: `enable = True`, `sequence_type = "XY4"`
- **`pt_options`** — Pauli twirling: `enable_gates = True`, `strategy = "active"`, `num_randomizations = 32`
- **`trex_options`** — TREX: `measure_mitigation = True`
- **`zne_options`** — ZNE with PEA: `zne_mitigation = True`, `noise_factors = [1, 3, 5]`, `amplifier = "pea"`
- **`pec_options`** — PEC: `pec_mitigation = True`, `max_overhead = 100`

</div>

In [ ]:
dd_options = EstimatorOptions()
# TODO: Add your code here

pt_options = EstimatorOptions()
# TODO: Add your code here

trex_options = EstimatorOptions()
# TODO: Add your code here

zne_options = EstimatorOptions()
# TODO: Add your code here

pec_options = EstimatorOptions()
# TODO: Add your code here


options_dict = {
    "dd": dd_options,
    "pt": pt_options,
    "trex": trex_options,
    "zne": zne_options,
    "pec": pec_options,
}


#### 2.2 Enhancing the Results of a Bell state estimation

We consider the Bell state

$$
\left\vert \Phi \right\rangle = \frac{1}{\sqrt{2}} \left( \left\vert 01 \right\rangle - \left\vert 10 \right\rangle\right)
$$

This state is an eigenstate of the observable

$$
H = XX + YY + ZZ
$$

such that
$$
H \left\vert \Phi \right\rangle = -3 \left\vert \Phi \right\rangle
$$

<div class="alert alert-block alert-success">

**Exercise 7 — Estimate ⟨H⟩ with and without error mitigation.**

1. Construct the `QuantumCircuit` `bell_circuit` that prepares the two-qubit Bell state $|\Phi\rangle$.
2. Construct the observable $H$ as a `SparsePauliOp` called `observable`.
3. Estimate $\langle H \rangle$ with an estimation primitive on three backends:
    1. an ideal `AerSimulator`;
    2. a noisy simulator built from the `ibm_kingston` noise model;
    3. the `ibm_kingston` QPU through the IBM Quantum API, with , with increasing levels of error suppression and mitigation.

**Note:** Runtime's *local testing mode* ignores error suppression and mitigation options, so in step 3.2 you may not see the effect of error mitigation and supression.

</div>

In [ ]:
# 1. Bell state |Φ⟩ = (|01⟩ - |10⟩)/√2
bell_circuit = QuantumCircuit(2)
# TODO: Add your code here

# 2. Observable H = XX + YY + ZZ
observable = None  # TODO: Add your code here

# 3a. Ideal AerSimulator
ideal_energy = None  # TODO: Add your code here

print(f"Ideal <H> = {ideal_energy:.4f}")

In [ ]:
# 3b. ibm_kingston noisy simulator
# TODO: Add your code here

noisy_energy = None  # TODO: Add your code here

print(f"Noisy <H> = {noisy_energy:.4f}")

In [ ]:
# 3c. ibm_kingston QPU with EM and/or ES techniques
hardware_options = EstimatorOptions()
# TODO: Add your code here


hardware_energy = None  # TODO: Add your code here

print(f"Hardware <H> = {hardware_energy:.4f}")

-----------

<a id="additional-information"></a>

# Additional information

Adapted from the Qiskit Global Summer School 2026 labs.

**Part A** created by Alberto Maldonado Romo & Gwonhak Lee.

**Part B** created by Sophy Shin & Sophie Engineer.

**Part B 2.2** section is created by Abdellah Tounsi.
